# trainedml - Quickstart

Ce notebook montre le workflow complet de **trainedml** : entraîner, évaluer, prédire, faire varier le seed et sauvegarder un modèle.

```bash
pip install trainedml
```

In [ ]:
from trainedml import Trainer

# Entraîner un KNN sur Iris (dataset intégré, chargé localement - aucun réseau requis).
# Le prétraitement (imputation + standardisation + one-hot) est appliqué automatiquement.
trainer = Trainer(dataset="iris", model="knn", model_params={"n_neighbors": 5})
trainer.fit()
trainer.evaluate()

## Varier le seed sans recréer le Trainer

`fit(seed=...)` re-splitte les données puis réentraîne : idéal pour vérifier la stabilité d'un modèle.

In [ ]:
for seed in range(5):
    scores = trainer.fit(seed=seed).evaluate()
    print(f"seed={seed} : accuracy={scores['accuracy']:.3f}")

## Prédire sur de nouvelles données

In [ ]:
trainer.predict([[5.1, 3.5, 1.4, 0.2], [6.2, 2.8, 4.8, 1.8]])

## Utiliser n'importe quel estimateur scikit-learn

Le `Trainer` accepte tout objet possédant `fit` et `predict` : l'écosystème scikit-learn entier fonctionne directement.

In [ ]:
from sklearn.svm import SVC

trainer_svc = Trainer(dataset="iris", model=SVC(kernel="rbf"))
trainer_svc.fit()
trainer_svc.evaluate()

## Sauvegarder et recharger un modèle

Le modèle **et** son préprocesseur sont sauvegardés ensemble : le modèle rechargé prédit directement sur des données brutes.

In [ ]:
trainer.save("model_iris.joblib")

restored = Trainer.load("model_iris.joblib")
restored.predict([[5.1, 3.5, 1.4, 0.2]])

## Données en mémoire (X, y)

Le `Trainer` accepte aussi vos propres DataFrames - la tâche (classification/régression) est détectée automatiquement et les métriques s'adaptent.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
X = pd.DataFrame({"a": rng.normal(size=200), "b": rng.uniform(0, 10, 200)})
y = pd.Series(3 * X["a"] + 0.5 * X["b"] + rng.normal(0, 0.1, 200), name="target")

reg = Trainer(X=X, y=y, model="ridge")
reg.fit()
reg.evaluate()  # métriques de régression : r2, mse, rmse, mae